In [1]:
from retrieval import PoliticalRAGRetriever
from evaluator import RAGRetrievalEvaluator

In [7]:
test_data_path = "../src/datasets/dataset_final_merged_with_mbfc_labels_without_duplicates_index_reset_anonymized_5_party_labels_media_labels_author_labels_stance_labels.csv"
ches_path = "../src/datasets/ground_truth/1999-2024_CHES.csv"

retriever_simple = PoliticalRAGRetriever(retrieval_mode="simple")
eval_simple = RAGRetrievalEvaluator(
    retriever=retriever_simple,
    test_data_path=test_data_path,
    ches_data_path=ches_path,
    sample_size=50,
    run_name="simple_baseline",
)

2026-05-18 15:27:42,082 - retrieval - INFO - Loading Bi-Encoder: paraphrase-multilingual-MiniLM-L12-v2
2026-05-18 15:27:42,084 - sentence_transformers.base.model - INFO - No device provided, using mps
/Users/janneslampe/Desktop/Coding/Master Thesis/Evaluation/.venv/lib/python3.14/site-packages/qdrant_client/qdrant_remote.py:288: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(
2026-05-18 15:27:42,337 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-18 15:27:42,378 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/modules.json "HTTP/1.1 200 OK"
2026-05-18 15:27:42,512 - httpx - INFO - HTTP Request: HEAD https:

In [49]:
site_content = eval_simple.df.iloc[0].site_content
full_text = eval_simple.df.iloc[0].full_text
party = eval_simple.df.iloc[0].party
year = int(eval_simple.df.iloc[0].year)

In [71]:
print(full_text)

#GroKo schaut tatenlos zu, wie sich Gesellschaft in #Corona-Krise weiter spaltet. Auf der einen Seite #Kurzarbeit, #Arbeitslosigkeit &amp; #Altersarmut. Und auf der anderen Seite immer mehr #Millionäre. #Vermögenssteuer für Superreiche &amp; #SozialeWende jetzt! 


In [37]:
retrieved_points = eval_simple.retriever.search(query=full_text)

2026-05-18 15:43:06,726 - httpx - INFO - HTTP Request: POST http://localhost:6333/collections/bundestag_speeches_chunks/points/query "HTTP/1.1 200 OK"


In [73]:
retrieved_points[0].score

0.66501206

In [ ]:
result_list = []

for i, point in enumerate(retrieved_points):
    results = {}
    results["pos"] = i + 1
    results["retrieved_party"] = point.payload.get("party", "")
    results["retrieved_speaker"] = point.payload.get("speaker", "")
    results["retrieved_date"] = point.payload.get("date", "")
    results["retrieved_year"] = int(results["retrieved_date"][:4])
    results["retrieval_score"] = point.payload.get("retrieval_score", 0.0)
    result_list.append(results)

In [65]:
result_list

[{'pos': 1,
  'retrieved_party': 'SPD',
  'retrieved_speaker': 'Dennis Rohde',
  'retrieved_date': '2020-09-29',
  'retrieved_year': 2020},
 {'pos': 2,
  'retrieved_party': 'DIE LINKE',
  'retrieved_speaker': 'Susanne Ferschl',
  'retrieved_date': '2020-11-25',
  'retrieved_year': 2020},
 {'pos': 3,
  'retrieved_party': 'DIE LINKE',
  'retrieved_speaker': 'Sabine Zimmermann',
  'retrieved_date': '2020-11-25',
  'retrieved_year': 2020}]

In [68]:
_, _, true_lrgen = eval_simple._get_closest_ches_score(party, year)
top_1_pred_lrgen = 0.0
predicted_lrgen = 0.0
for result in result_list:
    if result["pos"] == 1:
        _, _, top_1_pred_lrgen = eval_simple._get_closest_ches_score(
            result["retrieved_party"], int(result["retrieved_year"])
        )
    _, _, result_lrgen = eval_simple._get_closest_ches_score(
        result["retrieved_party"], int(result["retrieved_year"])
    )
    predicted_lrgen += result_lrgen
predicted_hits_lrgen = predicted_lrgen / len(result_list)

In [69]:
true_lrgen, top_1_pred_lrgen, predicted_hits_lrgen = (1.428571462631226, 3.61904764175415, 2.1587301890055337)

(1.428571462631226, 3.61904764175415, 2.1587301890055337)